<a href="https://colab.research.google.com/github/alexzdor/geo-reviews-sentiment/blob/main/notebooks/01_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("d0rj/geo-reviews-dataset-2023")
print(ds)

README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

data/train-00000-of-00001-49261e4e5a35a5(…): reconstructing file:   0%|          |  0.00B /  171MB            

data/train-00000-of-00001-49261e4e5a35a5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['address', 'name_ru', 'rating', 'rubrics', 'text'],
        num_rows: 500000
    })
})


In [ ]:
df = ds["train"].to_pandas()
print(df.shape)
df.head()

(500000, 5)


,address,name_ru,rating,rubrics,text
0,"Екатеринбург, ул. Московская / ул. Волгоградск...",Московский квартал,3,Жилой комплекс,Московский квартал 2.\nШумно : летом по ночам ...
1,"Московская область, Электросталь, проспект Лен...",Продукты Ермолино,5,Магазин продуктов;Продукты глубокой заморозки;...,"Замечательная сеть магазинов в общем, хороший ..."
2,"Краснодар, Прикубанский внутригородской округ,...",LimeFit,1,Фитнес-клуб,"Не знаю смутят ли кого-то данные правила, но я..."
3,"Санкт-Петербург, проспект Энгельса, 111, корп. 1",Snow-Express,4,Пункт проката;Прокат велосипедов;Сапсёрфинг,Хорошие условия аренды. \nДружелюбный персонал...
4,"Тверь, Волоколамский проспект, 39",Студия Beauty Brow,5,"Салон красоты;Визажисты, стилисты;Салон бровей...",Топ мастер Ангелина топ во всех смыслах ) Немн...


In [ ]:
print(df.columns)
print()
print(df["rating"].value_counts().sort_index())
a = sum(df["rating"].value_counts().sort_index())
for x in df["rating"].value_counts().sort_index():
  print(round(x/a * 100, 2))

Index(['address', 'name_ru', 'rating', 'rubrics', 'text'], dtype='object')

rating
0       200
1     34351
2     12088
3     21686
4     41160
5    390515
Name: count, dtype: int64
0.04
6.87
2.42
4.34
8.23
78.1


In [ ]:
print(df[["text", "rating"]].isna().sum())

text      0
rating    0
dtype: int64


In [ ]:
df["text_len"] = df["text"].str.len()
print(df["text_len"].describe())

count    500000.000000
mean        303.233766
std         292.550114
min           2.000000
25%         145.000000
50%         215.000000
75%         365.000000
max       20162.000000
Name: text_len, dtype: float64


In [ ]:
# Строки с рейтингом 0
print(df[df["rating"] == 0][["text", "rating"]].head(10))

                                                    text  rating
1907   Гостиница расположенна удобно, если отремонтир...       0
2165   Это мой любимый магазин с запчастями. Можно вс...       0
3584   Нет растительного масла для салата, нет зубочи...       0
3615   Все довольны, и взрослые и дети.\nВкусная еда....       0
6389   Большой выбор, очень грамотно и удобно разложе...       0
8232   Отличный гостевой дом. Тихо, чисто, удобные но...       0
15103  Рекомендую. \nХороший пляж. Лежаки есть (платн...       0
15554  Вкусно. Много народу, столик стоит бронировать...       0
18555  Вкусная кухня, приятная атмосфера. Понравилось...       0
20131  Там охранникам не хватает ума они из деревни т...       0


In [ ]:
# По 3 случайных примера каждой оценки
for r in [1, 2, 3, 4, 5]:
    print(f"--- Рейтинг {r} ---")
    for t in df[df["rating"] == r]["text"].sample(3, random_state=42):
        print(t[:300], "\n")

--- Рейтинг 1 ---
Водил жену, грустно всё это. Перца нет, мяса нет, совести тоже, выглядит, как в дешевой столовке. Очень грязно.  

Никогда не писала отзывы, но тут я просто не смогла пройти мимо. Не дай бог кому-то попасть в этот филиал ада. \nГоспитализировали маму с механической желтухой, 2 недели мариновали, никуда не спешат ребятки там. За 2 недели было сделано узи, кровь на биохимию, мрт и фгдс (когда все это можно за 2 дн 

По итогам обращения в это заведение, мы потеряли кота… Если вы любите своего пушистика, вам не сюда! \nОбратились в клинику с очередным обострением дерматита у кота. Врач, Денис Андреевич, назначил препарат для лечения. Спустя 2 недели у кота впервые в жизни резко обострился цистит. При повторном об 

--- Рейтинг 2 ---
Часто летаю этой авиакомпанией.\nПо мимо всяких коллизий с багажом и сервисом... был очень удивлен тем, что по правилам авиакомпании при перевозке личного оружия (ружье или пистолет) вы должны их уведомить за 90 дней и иметь билет на руках... 

In [ ]:
df = df[df["rating"] > 0][["text", "rating"]].copy()
print(df.shape)

(499800, 2)


In [ ]:
def to_label(r):
    if r <= 2:
        return 0   # негатив
    elif r == 3:
        return 1   # нейтрально
    else:
        return 2   # позитив

df["label"] = df["rating"].apply(to_label)
print(df["label"].value_counts().sort_index())

label
0     46439
1     21686
2    431675
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
)
print(len(train_df), len(val_df), len(test_df))

399840 49980 49980


In [ ]:
#Проверка стратификации
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, part["label"].value_counts(normalize=True).sort_index().round(4).to_dict())

train {0: 0.0929, 1: 0.0434, 2: 0.8637}
val {0: 0.0929, 1: 0.0434, 2: 0.8637}
test {0: 0.0929, 1: 0.0434, 2: 0.8637}


In [ ]:
#Балансировка train
import pandas as pd
def balance(df, n_per_class, seed):
    parts = [
        df[df["label"] == c].sample(n_per_class, random_state=seed)
        for c in sorted(df["label"].unique())
    ]
    return pd.concat(parts).sample(frac=1, random_state=seed)

n_min = train_df["label"].value_counts().min()
train_bal = balance(train_df, n_min, seed=42)
print(train_bal["label"].value_counts())

label
1    17349
0    17349
2    17349
Name: count, dtype: int64


In [ ]:
#Baseline
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_features=200_000,
    sublinear_tf=True,
)
X_train = vectorizer.fit_transform(train_bal["text"])
X_val = vectorizer.transform(val_df["text"])

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, train_bal["label"])

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
from sklearn.metrics import classification_report, f1_score

preds = clf.predict(X_val)
print(classification_report(val_df["label"], preds, digits=3))
print("macro-F1:", f1_score(val_df["label"], preds, average="macro"))

              precision    recall  f1-score   support

           0      0.711     0.759     0.734      4644
           1      0.199     0.685     0.308      2168
           2      0.992     0.864     0.924     43168

    accuracy                          0.846     49980
   macro avg      0.634     0.769     0.656     49980
weighted avg      0.932     0.846     0.879     49980

macro-F1: 0.655509870926943


In [ ]:
import numpy as np

proba = clf.predict_proba(X_val)   # матрица (n_объектов, 3) вероятностей

In [ ]:
train_prior = np.array([1/3, 1/3, 1/3])
real_prior = train_df["label"].value_counts(normalize=True).sort_index().values

w = real_prior / train_prior
preds_adj = (proba * w).argmax(axis=1)

print(classification_report(val_df["label"], preds_adj, digits=3))
print("macro-F1:", f1_score(val_df["label"], preds_adj, average="macro"))

              precision    recall  f1-score   support

           0      0.837     0.662     0.739      4644
           1      0.395     0.091     0.148      2168
           2      0.935     0.992     0.962     43168

    accuracy                          0.922     49980
   macro avg      0.722     0.582     0.617     49980
weighted avg      0.902     0.922     0.906     49980

macro-F1: 0.6166987360667139


In [ ]:
for alpha in [0, 0.25, 0.5, 0.75, 1.0]:
    p = (proba * (w ** alpha)).argmax(axis=1)
    print(alpha, round(f1_score(val_df["label"], p, average="macro"), 4))

0 0.6555
0.25 0.6876
0.5 0.699
0.75 0.6766
1.0 0.6167


In [ ]:
import re

def tokenize(text):
    return re.findall(r"\w+", text.lower())

print(tokenize("Не плохо, но дорого!"))

['не', 'плохо', 'но', 'дорого']


In [ ]:
from collections import Counter

counter = Counter()
for text in train_bal["text"]:
    counter.update(tokenize(text))

min_freq = 3
itos = ["<PAD>", "<UNK>"] + [w for w, c in counter.items() if c >= min_freq]
stoi = {w: i for i, w in enumerate(itos)}
print(len(itos))


53439


In [ ]:
import numpy as np

lens = train_bal["text"].apply(lambda t: len(tokenize(t)))
print(lens.describe())
print({p: int(np.percentile(lens, p)) for p in [50, 90, 95, 99]})

count    52047.00000
mean        62.93917
std         66.47985
min          1.00000
25%         25.00000
50%         43.00000
75%         78.00000
max       3120.00000
Name: text, dtype: float64
{50: 43, 90: 131, 95: 177, 99: 310}


In [ ]:
MAX_LEN = 180

def encode(text):
    ids = [stoi.get(w, 1) for w in tokenize(text)]   # 1 = <UNK>
    return ids[:MAX_LEN]                              # truncation

In [ ]:
print(encode("Не плохо, но дорого!"))
print(len(encode(train_bal["text"].iloc[0])))

[43, 2074, 63, 1063]
32


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.ids = [encode(t) for t in texts]
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.ids[i], self.labels[i]

In [ ]:
train_ds = ReviewDataset(train_bal["text"], train_bal["label"])
val_ds = ReviewDataset(val_df["text"], val_df["label"])
print(len(train_ds), len(val_ds))
print(train_ds[0])

52047 49980
([2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 12, 1, 17, 18, 19, 20, 21, 22, 8, 23, 24, 25, 26, 27, 28, 29, 8], 1)


In [ ]:
def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    len_max = lengths.max().item()
    padded = torch.tensor([s + [0] * (len_max - len(s)) for s in seqs])
    labels = torch.tensor(labels)
    return padded, lengths, labels

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, collate_fn=collate_fn)

padded, lengths, labels = next(iter(train_loader))
print(padded.shape, lengths[:10], labels[:10])

torch.Size([64, 180]) tensor([64, 21, 24, 46, 24, 66, 14, 20, 62, 92]) tensor([1, 2, 0, 2, 2, 0, 2, 1, 2, 1])


In [ ]:
#модель
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, n_classes=3, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        out = self.dropout(h[-1])
        return self.fc(out)

In [ ]:
model = LSTMClassifier(vocab_size=len(itos))
logits = model(padded, lengths)
print(logits.shape)
print(logits[0])
print(sum(p.numel() for p in model.parameters()))

torch.Size([64, 3])
tensor([0.0896, 0.1903, 0.0417], grad_fn=<SelectBackward0>)
6972675


In [ ]:
import torch
z = torch.tensor([-1.2, 0.3, 2.1])
print(torch.softmax(z, dim=0))
print(torch.softmax(z + 100, dim=0))   # прибавили 100 ко всем

tensor([0.0307, 0.1375, 0.8318])
tensor([0.0307, 0.1375, 0.8318])


In [ ]:
criterion = nn.CrossEntropyLoss()

loss = criterion(logits, labels)
print(loss.item())

1.1106762886047363


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = LSTMClassifier(vocab_size=len(itos)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for padded, lengths, labels in loader:
        padded, labels = padded.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(padded, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for padded, lengths, labels in loader:
        logits = model(padded.to(device), lengths)
        all_logits.append(logits.cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

In [ ]:
from sklearn.metrics import f1_score

for epoch in range(3):
    tr_loss = train_epoch(model, train_loader)
    logits, labels = evaluate(model, val_loader)
    preds = logits.argmax(dim=1)
    f1 = f1_score(labels, preds, average="macro")
    print(f"epoch {epoch+1}: train_loss={tr_loss:.4f}, val macro-F1={f1:.4f}")

epoch 1: train_loss=0.8131, val macro-F1=0.5948
epoch 2: train_loss=0.6048, val macro-F1=0.6236
epoch 3: train_loss=0.5040, val macro-F1=0.6169


In [ ]:
import copy

best_f1, best_state = 0, None

for epoch in range(8):
    # новая сбалансированная выборка каждую эпоху
    ep_df = balance(train_df, n_min, seed=epoch)
    ep_ds = ReviewDataset(ep_df["text"], ep_df["label"])
    ep_loader = DataLoader(ep_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)

    tr_loss = train_epoch(model, ep_loader)
    logits, labels = evaluate(model, val_loader)
    f1 = f1_score(labels, logits.argmax(dim=1), average="macro")
    print(f"epoch {epoch+1}: train_loss={tr_loss:.4f}, val macro-F1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_state = copy.deepcopy(model.state_dict())

epoch 1: train_loss=0.4936, val macro-F1=0.6598
epoch 2: train_loss=0.4447, val macro-F1=0.6575
epoch 3: train_loss=0.4010, val macro-F1=0.6587
epoch 4: train_loss=0.3626, val macro-F1=0.6567
epoch 5: train_loss=0.3220, val macro-F1=0.6691
epoch 6: train_loss=0.2858, val macro-F1=0.6734
epoch 7: train_loss=0.2667, val macro-F1=0.6504
epoch 8: train_loss=0.2423, val macro-F1=0.6702


In [ ]:
model.load_state_dict(best_state)
logits, labels = evaluate(model, val_loader)
proba = torch.softmax(logits, dim=1).numpy()

In [ ]:
for alpha in [0, 0.25, 0.5, 0.75, 1.0]:
    p = (proba * (w ** alpha)).argmax(axis=1)
    print(alpha, round(f1_score(labels, p, average="macro"), 4))

0 0.6734
0.25 0.6789
0.5 0.6845
0.75 0.683
1.0 0.6679


In [ ]:
conf = proba.max(axis=1)                       # уверенность модели
correct = (proba.argmax(axis=1) == labels.numpy())   # угадала или нет

bins = np.linspace(0.3, 1.0, 8)                # границы корзин
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (conf >= lo) & (conf < hi)
    if mask.sum() > 0:
        print(f"{lo:.1f}-{hi:.1f}: n={mask.sum():5d}, "
              f"уверенность={conf[mask].mean():.3f}, точность={correct[mask].mean():.3f}")

0.3-0.4: n=   84, уверенность=0.378, точность=0.393
0.4-0.5: n=  679, уверенность=0.465, точность=0.429
0.5-0.6: n= 1587, уверенность=0.550, точность=0.474
0.6-0.7: n= 1623, уверенность=0.651, точность=0.484
0.7-0.8: n= 1964, уверенность=0.753, точность=0.532
0.8-0.9: n= 2992, уверенность=0.855, точность=0.624
0.9-1.0: n=41051, уверенность=0.987, точность=0.965


In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

text = "Не плохо, но дорого! Тараканище в номере..."
print(tok.tokenize(text))
print(tok(text))

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

['Не', 'плохо', ',', 'но', 'дорого', '!', 'Тара', '##кани', '##ще', 'в', 'номере', '.', '.', '.']
{'input_ids': [2, 6226, 30350, 16, 1363, 39673, 5, 81492, 29024, 6139, 314, 43463, 18, 18, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
print(tok.convert_ids_to_tokens([2, 3]))
print(tok.cls_token, tok.sep_token, tok.pad_token_id)

['[CLS]', '[SEP]']
[CLS] [SEP] 0


In [ ]:
lens_bert = [len(tok(t)["input_ids"]) for t in train_bal["text"].sample(5000, random_state=0)]
print({p: int(np.percentile(lens_bert, p)) for p in [50, 90, 95, 99]})

{50: 66, 90: 194, 95: 269, 99: 503}


In [ ]:
enc = tok(
    list(train_bal["text"][:3]),
    truncation=True,
    max_length=256,
    padding=True,
    return_tensors="pt",
)
print(enc["input_ids"].shape)
print(enc["attention_mask"])

torch.Size([3, 100])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
from transformers import AutoModelForSequenceClassification

bert = AutoModelForSequenceClassification.from_pretrained(
    "cointegrated/rubert-tiny2", num_labels=3
)
print(sum(p.numel() for p in bert.parameters()))

model.safetensors: reconstructing file:   0%|          |  0.00B /  118MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


29194707


In [ ]:
class BertDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.texts[i], self.labels[i]

def bert_collate(batch):
    texts, labels = zip(*batch)
    enc = tok(list(texts), truncation=True, max_length=256,
              padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels)
    return enc

In [ ]:
bert_val_loader = DataLoader(BertDataset(val_df["text"], val_df["label"]),
                             batch_size=128, shuffle=False, collate_fn=bert_collate)

In [ ]:
bert = bert.to(device)
optimizer = torch.optim.AdamW(bert.parameters(), lr=2e-5, weight_decay=0.01)

def train_epoch_bert(model, loader):
    model.train()
    total = 0
    for batch in loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        logits = model(**batch).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(labels)
    return total / len(loader.dataset)

@torch.no_grad()
def evaluate_bert(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        labels = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        all_logits.append(model(**batch).logits.cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

In [ ]:
best_f1, best_state = 0, None

for epoch in range(3):
    ep_df = balance(train_df, n_min, seed=100 + epoch)
    ep_loader = DataLoader(BertDataset(ep_df["text"], ep_df["label"]),
                           batch_size=32, shuffle=True, collate_fn=bert_collate)
    tr_loss = train_epoch_bert(bert, ep_loader)
    logits, labels = evaluate_bert(bert, bert_val_loader)
    f1 = f1_score(labels, logits.argmax(dim=1), average="macro")
    print(f"epoch {epoch+1}: train_loss={tr_loss:.4f}, val macro-F1={f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_state = copy.deepcopy(bert.state_dict())

epoch 1: train_loss=0.5979, val macro-F1=0.6524
epoch 2: train_loss=0.5011, val macro-F1=0.6799
epoch 3: train_loss=0.4627, val macro-F1=0.6843


In [ ]:
bert.load_state_dict(best_state)
logits, labels = evaluate_bert(bert, bert_val_loader)
proba_bert = torch.softmax(logits, dim=1).numpy()

for alpha in [0, 0.25, 0.5, 0.75, 1.0]:
    p = (proba_bert * (w ** alpha)).argmax(axis=1)
    print(alpha, round(f1_score(labels, p, average="macro"), 4))

0 0.6843
0.25 0.7012
0.5 0.7173
0.75 0.7258
1.0 0.7091


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# TF-IDF: вероятности у нас в переменной proba от LogReg, если вы её не перезаписали
# LSTM: если proba перезаписана, пересчитайте через model.load_state_dict(...)
alpha = 0.5
p_bert = (proba_bert * (w ** alpha)).argmax(axis=1)
print("BERT по классам:", f1_score(labels, p_bert, average=None).round(3))

BERT по классам: [0.798 0.393 0.961]


In [ ]:
proba_tfidf = clf.predict_proba(X_val)
p_tfidf = (proba_tfidf * (w ** alpha)).argmax(axis=1)
print("TF-IDF по классам:", f1_score(val_df["label"], p_tfidf, average=None).round(3))

TF-IDF по классам: [0.778 0.354 0.965]


In [ ]:
from sklearn.metrics import confusion_matrix

p_best = (proba_bert * (w ** 0.75)).argmax(axis=1)
print(confusion_matrix(labels, p_best))

[[ 3827   593   224]
 [  585  1009   574]
 [  443  1263 41462]]


In [ ]:
val_reset = val_df.reset_index(drop=True)
val_reset["pred"] = p_best
val_reset["conf"] = proba_bert.max(axis=1)

# уверенные ошибки на нейтральном классе
errs = val_reset[(val_reset["label"] == 1) & (val_reset["pred"] != 1)]
for _, r in errs.sort_values("conf", ascending=False).head(8).iterrows():
    print(f"рейтинг={r['rating']}, предсказано={r['pred']}, conf={r['conf']:.2f}")
    print(r["text"][:300], "\n")

рейтинг=3, предсказано=2, conf=0.99
Удобное по расположению место! Чистое, приятное, светлое помещение. Имеются все принадлежности, чтобы привести себя в порядок перед самим фото! Меня принимала  София, добрый и отзывчивый фотограф) Она вошла в мое положение и создала для меня комфортную атмосферу. \nМесто прекрасное! Советую!  

рейтинг=3, предсказано=2, conf=0.99
Два года подряд, отмечаем новый год компанией, кухня хорошая готовят вкусно, алкоголь,  коктейли вино всё хорошего качества, мы все остались довольны, музыкальная подборка, класса!!! Всё супер. 

рейтинг=3, предсказано=2, conf=0.99
Александра очень доброжелательна, мастер своего дела! Сделала мне красивый макияж и прическу, мне очень понравились! Также удобное месторасположение и очень комфортная обстановка. Рекомендую!!! 

рейтинг=3, предсказано=2, conf=0.99
Останавливались на ночлег 1 июня. Замечательный мини- отель. Отличные номера. Приветливый персонал. Теперь всегда буду останавливаться здесь  

рейтинг=3, предсказано=2

In [ ]:
errs = val_reset[(val_reset["label"] == 1) & (val_reset["pred"] != 1)]
sample = errs.sample(20, random_state=0)
names = ["негатив", "нейтральный", "позитив"]
for _, r in sample.iterrows():
    print(f"рейтинг={r['rating']}, модель сказала: {names[r['pred']]}, conf={r['conf']:.2f}")
    print(r["text"][:300], "\n")

рейтинг=3, модель сказала: позитив, conf=0.54
Хорошие виды есть выход на речку просто прогуляться приехать не вариант потому что находится слишком далеко от центра просто приехать на метро встречного вокзала потратить на это кучу времени чтобы просто погулять зимой вообще ничего не было просто приехали посмотреть на её супер развлечения на втор 

рейтинг=3, модель сказала: негатив, conf=0.57
Сам музей прекрасен, историческое знаковое место для всех, кто почитает творчество Лермонтова. Но администраторы музея очень неприятные люди и крайне необязательные. Была в музее несколько раз, и всегда одно и тоже .   Звонишь- обещают экскурсию  и подробно все рассказывают, как и когда, приезжаешь  

рейтинг=3, модель сказала: негатив, conf=0.65
На компанию брали более десяти стандартных номеров. По одному номеру более чем за неделю уведомили о заезде семьи 2 взрослых и 1 ребенок 15 лет за доплату.  В итоге в однокомнатном номере была поставлена раскладушка. Все бы ничего, пока не узнали, что част

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

ALPHA = {"tfidf": 0.5, "lstm": 0.5, "bert": 0.75}
torch.save(best_state, "/content/drive/MyDrive/bert_best.pt")

Mounted at /content/drive


In [ ]:
bert_test_loader = DataLoader(BertDataset(test_df["text"], test_df["label"]),
                              batch_size=128, shuffle=False, collate_fn=bert_collate)
bert.load_state_dict(best_state)
logits, y_test = evaluate_bert(bert, bert_test_loader)
proba_t = torch.softmax(logits, dim=1).numpy()
p = (proba_t * (w ** ALPHA["bert"])).argmax(axis=1)
print("BERT test macro-F1:", round(f1_score(y_test, p, average="macro"), 4))

BERT test macro-F1: 0.7243


In [ ]:
# TF-IDF
X_test = vectorizer.transform(test_df["text"])          # только transform!
proba_t = clf.predict_proba(X_test)
p = (proba_t * (w ** ALPHA["tfidf"])).argmax(axis=1)
print("TF-IDF test macro-F1:", round(f1_score(test_df["label"], p, average="macro"), 4))

TF-IDF test macro-F1: 0.7045


In [ ]:
model = LSTMClassifier(vocab_size=len(itos)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_f1, best_state_lstm = 0, None
for epoch in range(8):
    ep_df = balance(train_df, n_min, seed=epoch)
    ep_ds = ReviewDataset(ep_df["text"], ep_df["label"])
    ep_loader = DataLoader(ep_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)

    tr_loss = train_epoch(model, ep_loader)
    logits, labels = evaluate(model, val_loader)
    f1 = f1_score(labels, logits.argmax(dim=1), average="macro")
    print(f"epoch {epoch+1}: train_loss={tr_loss:.4f}, val macro-F1={f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_state_lstm = copy.deepcopy(model.state_dict())

torch.save(best_state_lstm, "lstm_best.pt")

epoch 1: train_loss=0.7856, val macro-F1=0.6003
epoch 2: train_loss=0.6019, val macro-F1=0.6472
epoch 3: train_loss=0.5353, val macro-F1=0.6465
epoch 4: train_loss=0.4791, val macro-F1=0.6367
epoch 5: train_loss=0.4301, val macro-F1=0.6739
epoch 6: train_loss=0.3823, val macro-F1=0.6748
epoch 7: train_loss=0.3386, val macro-F1=0.6732
epoch 8: train_loss=0.3089, val macro-F1=0.6635


In [ ]:
model.load_state_dict(best_state_lstm)
logits, labels = evaluate(model, val_loader)
proba_v = torch.softmax(logits, dim=1).numpy()
for alpha in [0, 0.25, 0.5, 0.75, 1.0]:
    p = (proba_v * (w ** alpha)).argmax(axis=1)
    print(alpha, round(f1_score(labels, p, average="macro"), 4))

0 0.6748
0.25 0.6847
0.5 0.6879
0.75 0.6757
1.0 0.6446


In [ ]:
# LSTM
test_loader = DataLoader(ReviewDataset(test_df["text"], test_df["label"]),
                         batch_size=256, shuffle=False, collate_fn=collate_fn)
model.load_state_dict(best_state_lstm)   # веса лучшей эпохи LSTM
logits, y_test = evaluate(model, test_loader)
proba_t = torch.softmax(logits, dim=1).numpy()
p = (proba_t * (w ** ALPHA["lstm"])).argmax(axis=1)
print("LSTM test macro-F1:", round(f1_score(y_test, p, average="macro"), 4))

LSTM test macro-F1: 0.6946
